# Titanic Survival: A Reproducible Data Workflow

**Name**: Tobias Frei Data **Dataset:** [Titanic - Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic/data) (Kaggle), training split `train.csv` with 891 passengers and 12 columns


**Project Description**: This notebook is a complete workflow that is reproducible. It loads, cleans, explores and visualizes the passanger data from the Titanic. It shows how the survival rate amongst passengers is distributed and investigates how survival rate behaves for different markers like sex, passenger class and age, and how handling of missing ages for passenger changes what the data appears to show. This workflow is intended to serve as a foundation for later machine learning models.

## Table of Content
1. Introduction
2. Data Ingestion
3. Data Familiarization
4. Research Questions
5. Data Wrangling
6. Exploratory Data Analysis
7. Summary

---

## 1. Introduction
The Titanic passenger dataset from Kaggle, provides historical records of passengers and their survival outcomes following the sinking of the Titanic in 1912.

The iconic principle of **“Women and Children First,”** which origin dates back to this disaster, forms the basis of my hypothesis: **Women and children had higher survival rates because they were given priority during lifeboat evacuation.**

My analysis aims to evaluate this hypothesis using historical passenger data.

### Motivation
I chose this dataset because it is small enough to check every column by hand, yet it has the problems a cleaning workflow has to handle in real data: 
- 136 of the 177 missing ages are in third class
- the cabin column is 3/4 empty
- fare was paid per ticket, not per person. 

That all together makes it a good test case for a workflow in which every cleaning decision is visible and can be repeated and it shows the importance of datawrangling prior to the actual training and building of an ML-model to predict survival rate!



### Scope
Only the Kaggle training split (train.csv) is used. 
The test split (test.csv) has no Survived column and is deliberately left untouched, so that it remains unseen data for a future model.

---

## 2. Data Ingestion

In [16]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
# Load the dataset
df = pd.read_csv('train.csv')

In [36]:
# Keep a raw copy of the dataset before any modifications as a backup
df_raw = df

---

## 3. Data Familiarization

My approach is to get a clear picture of every column before changing anything. 
First I used the standard pd-commands to look at the structure, then I check the missing values in detail.

### General descriptive statistics

In [33]:
# Display the first few rows of the dataset
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [37]:
# Data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [ ]:
# Descriptive statistics for the numeric columns
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [38]:
# Descriptive statistics for the non-numeric columns
df.describe(exclude="number")

,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


### Missing Values

In [ ]:
# Count of missing values for each column
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [39]:
# Percentage of missing values for each column
df.isna().mean().mul(100).round(1)

PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age            19.9
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
Cabin          77.1
Embarked        0.2
dtype: float64

#### Reusable function for missing values
Since missing values are one of the biggest challenge in this dataset I will create a reusable function for display purposes, to check before and after cleaning the data

In [42]:
def profile_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Returns a table containing data for dtype missing values per column and percentage of missing values, ordered by most gaps first"""
    summary = pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing": df.isna().sum(),
            "missing_pct": (df.isna().mean() * 100).round(1),
        }
    )
    return summary.sort_values("missing", ascending=False)   

In [43]:
# Function in Action: Profile missing values in the DataFrame
profile_missing(df)

,dtype,missing,missing_pct
Cabin,str,687,77.1
Age,float64,177,19.9
Embarked,str,2,0.2
PassengerId,int64,0,0.0
Name,str,0,0.0
Pclass,int64,0,0.0
Survived,int64,0,0.0
Sex,str,0,0.0
Parch,int64,0,0.0
SibSp,int64,0,0.0


In [50]:
# Lets check for dublicates
print(df.duplicated().sum())

# If there are duplicates, lets inspect them
df[df.duplicated()]

0


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked


In [44]:
# Here I check for missing values in the "Age" column grouped by "Pclass"
pd.crosstab(df["Pclass"], df["Age"].isna(), margins=True)


Age,False,True,All
Pclass,,,
1,186,30,216
2,173,11,184
3,355,136,491
All,714,177,891


In [58]:
# The same for Cabin: counts, then the share within each class in % (True = cabin missing)
display(pd.crosstab(df["Pclass"], df["Cabin"].isna(), margins=True))
(pd.crosstab(df["Pclass"], df["Cabin"].isna(), normalize="index") * 100).round(1)


Cabin,False,True,All
Pclass,,,
1,176,40,216
2,16,168,184
3,12,479,491
All,204,687,891


Cabin,False,True
Pclass,,
1,81.5,18.5
2,8.7,91.3
3,2.4,97.6


In [46]:
# While having a closer look at the "Ticket" column, I check the most common values and their frequencies, since I read that the fare was not necessarily tied to only one ticket number
df["Ticket"].value_counts().head()


Ticket
347082      7
1601        7
CA. 2343    7
3101295     6
CA 2144     6
Name: count, dtype: int64

In [47]:
# For purpose of examining this further lets take the ticket with the highest frequency
df.loc[df["Ticket"] == "347082", ["Name", "Pclass", "Fare"]]


,Name,Pclass,Fare
13,"Andersson, Mr. Anders Johan",3,31.275
119,"Andersson, Miss. Ellis Anna Maria",3,31.275
541,"Andersson, Miss. Ingeborg Constanzia",3,31.275
542,"Andersson, Miss. Sigrid Elisabeth",3,31.275
610,"Andersson, Mrs. Anders Johan (Alfrida Konstant...",3,31.275
813,"Andersson, Miss. Ebba Iris Alfrida",3,31.275
850,"Andersson, Master. Sigvard Harald Elias",3,31.275


In [54]:
# Is Fare a price per person or per ticket? 
# Median fare in third class by the number of passengers on the ticket
third = df[df["Pclass"] == 3]
# Map the number of passengers on each ticket to the third class dataframe
passengers_on_ticket = third["Ticket"].map(df["Ticket"].value_counts()).rename("passengers_on_ticket")
# Group by the number of passengers on the ticket and calculate the median fare
third.groupby(passengers_on_ticket)["Fare"].median()

passengers_on_ticket
1     7.89580
2    15.79585
3    22.33750
4    23.27085
5    29.12500
6    39.68750
7    56.49580
Name: Fare, dtype: float64

In [55]:
# How often does each value occur in the categorical columns?
for column in ["Survived", "Pclass", "Sex", "Embarked"]:
    display(df[column].value_counts())

Survived
0    549
1    342
Name: count, dtype: int64

Pclass
3    491
1    216
2    184
Name: count, dtype: int64

Sex
male      577
female    314
Name: count, dtype: int64

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [56]:
# The same as shares in % (missing values are not counted)
for column in ["Survived", "Pclass", "Sex", "Embarked"]:
    display((df[column].value_counts(normalize=True) * 100).round(1))

Survived
0    61.6
1    38.4
Name: proportion, dtype: float64

Pclass
3    55.1
1    24.2
2    20.7
Name: proportion, dtype: float64

Sex
male      64.8
female    35.2
Name: proportion, dtype: float64

Embarked
S    72.4
C    18.9
Q     8.7
Name: proportion, dtype: float64

---

## Dataset Overview in Detail
The training dataset contains 891 passengers and 12 columns, one row per passenger, with no duplicate rows. 38.4 % of the passengers survived.

**Identifier**
- **PassengerId** (int) Row number from the Kaggle training dataset, 1 to 891. This has no analytical meaning but good to be used to index later on.

**Target**
- **Survived** (int) 1 = survived, 0 = died.

**Passenger attribute variables**
These variables can be used to predict the target later in ML-models.
- **Pclass** (int) Ticket class: 1 = first - 2 = second - 3 = third thier ticket class. The median is 3, so at least half of all of the passengers travelled third class
- **Name** (str) Format "Surname followed by Title. Given names"
- **Sex** (str) 577 male passengers and 314 female passengers.
- **Age** (float) 0.42 to 80 years, median 28. **177 values are missing (19.9 %), 136 of them in third class. This is significant amount of missing age gap in the 3. tier**
- **SibSp** (int) Number of siblings and married couples.
- **Parch** (int) Number of parents and childrens.

**Ticket and travel information**
- **Ticket** (str) Only 681 different values for 891 passengers, so some passengers share a ticket or have a group ticket. Needs further investigation.
- **Fare** (float) Price of the ticket. This column is skewed: median 14.45 - mean 32.20 - maximum 512.33 and some fares are 0. This needs further investigation if it was actually free or issue with data collection. Assumption: Shared tickets show the full price for every passenger: the seven members of the Andersson family all show 31.275 for example.
- **Cabin** (str) Cabin number. **687 values are missing (77.1 %)**, but very unevenly: 18.5 % in first class, 91.3 % in second and 97.6 % in third class. Assumption: Not all classes actually had cabines. This is potentially being explained by mass cabines for 2./3. tier. 
- **Embarked** (str) Port of where they boarded: S = Southampton, C = Cherbourg, Q = Queenstown. **2 values are missing.**

---

## 4. Research Question

The familiarization showed that `Survived` is a clear target, that `Sex` and `Pclass` have no missing values, and that `Age` has gaps that are handled in Section 4. 

These three variables connect to the iconic questions about the Titanic: did "women and children first" hold, and additionally did first-class passengers have better chances? 

The analysis therefore asks two questions, one about sex and class and one about age. `Fare` and `Cabin` are not used, because the fare is a price per ticket and the cabin is mostly missing (see Section 2).


### Research Question 1
**How did the survival rate differ by sex and passenger class?**

- Dependent variable: `Survived`

- Independent variable 1: `Sex` Did "women first" actually turn into higher survival rates for women?

- Independent variable 2: `Pclass` Did first-class passengers have a better chance on surviving.

### Research Question 2
**How was age related to survival?**

- Dependent variable: `Survived`

- Independent variable: `Age` Were children more likely to survive?

Note: Only the observed ages, so only passengers with a not NaN value in the age column, are used for this question.
A strategy to handle missing values is to impute the missing datapoints with the median. So for the 177, I will impute ages, so I will use the median of each class and sex group. 
The caveat: they will pile up at a few values and if I would use them for the analysis they would distort the effect of class and sex into the age groups instead of showing a real effect of age, so they are not used for answering the age-related research-questions.



---

## 5. Data Wrangling

### Key Data Quality Issues
1. **Age** is missing for 177 passengers mostly in the 3rd class, so the gaps dont appear to be random. Research Question 2 will use only the observed ages, but a later model needs a complete Age column, so the gaps are filled and marked.
2. **Cabin** is missing for 77.1 % of passengers, too many gaps to fill in a meaningful way. This is ok since we dont need this information to answer our research question, for further analysis this needs to be adressed with the data collection team.
3. **Embarked** is missing for 2 passengers.
4. **Fare** is a price per ticket, not the price per person. Fare is not part of my research questions, so I keep it as recorded.

### Data Cleaning Strategy
I do not drop any rows. Dropping for example passengers without an age would remove mostly 3rd class passengers and distort the comparison between the classes. 
Instead I fill the missing ages with the median age of passengers with the same class and sex, and a flag column marks which ages were imputed. 
The two missing ports are filled with the most frequent port. The cabin number is mostly unknown. I do not know if they did not record the cabin, or if lower class passengers just did not have a cabin at all, but whether it was recorded is still information, so I replace `Cabin` with a yes/no column. I delibaretly chose to replace the cabin number with a true/false value, as the cabin number does not hold any value to our analysis.

In [60]:
# Median age per class and sex: These values will be filled into the gaps.
df.groupby(["Pclass", "Sex"])["Age"].median()

# Note: The median ignores missing values, and grouping by class and sex makes it a fair estimate, because each group still has at least 74 observed ages and the missing ages are assumed to be similar to the observed ones within the same group, but this is an assumption and may not hold in all cases.


Pclass  Sex   
1       female    35.0
        male      40.0
2       female    28.0
        male      30.0
3       female    21.5
        male      25.0
Name: Age, dtype: float64

#### Observation
The median age ranges from 21.5 (women in third class) to 40 (men in first class). One overall median of 28 would be a bad estimate for most groups, so I fill the gaps per group.

In [61]:
## Building the data cleaning pipeline functions
# 1. Impute Age and Fill with Median
def impute_age(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missed ages with the median age of the same class and sex.

    The column age_was_missing flags which age values were estimated.
    """
    out = df.copy()
    out["age_was_missing"] = out["Age"].isna()
    out["Age"] = out["Age"].fillna(out.groupby(["Pclass", "Sex"])["Age"].transform("median"))
    return out

In [62]:
# 2. Fill missing values with the mode (most frequent value)
def fill_with_mode(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Fill missing values in a column with its most frequent value."""
    out = df.copy()
    out[column] = out[column].fillna(out[column].mode().iloc[0])
    return out

In [ ]:
# 3. Replace the Cabin with the new flag Column has_cabin by checking if Cabin is not empty and dropping the original Cabin column
def replace_cabin_with_flag(df: pd.DataFrame) -> pd.DataFrame:
    """Replace the mostly empty Cabin column with a True/False column has_cabin."""
    out = df.copy()
    out["has_cabin"] = out["Cabin"].notna()
    return out.drop(columns="Cabin")

---

## 6. Exploratory Data Analysis

---

## 7. Summary